# NLP Mastery Journey — Module 4: Word & Sentence Embeddings, In Depth

In Module 3 you got a *preview* of embeddings. Now we go under the hood: how Word2Vec/GloVe/FastText are actually trained, how to explore and visualize the resulting vector space, how to go from word vectors to document vectors properly, and how modern sentence embeddings (built on transformers) supersede all of it for many production use cases.

### What this notebook teaches
| # | Topic | Why it matters |
|---|-------|------------------|
| 1 | The distributional hypothesis | The one idea every embedding method is built on |
| 2 | Word2Vec internals (CBOW vs Skip-gram, negative sampling) | So "just call `.fit()`" stops being a black box |
| 3 | GloVe | Global co-occurrence statistics, count-based vs. predict-based |
| 4 | FastText | Subword embeddings — the fix for out-of-vocabulary words |
| 5 | Exploring the vector space | Similarity, analogies, PCA/t-SNE visualization |
| 6 | Word vectors → document vectors | Averaging, TF-IDF-weighted averaging, Doc2Vec |
| 7 | Sentence embeddings (transformer-based) | The modern default; pooling strategies explained |
| 8 | Evaluation | Intrinsic vs. extrinsic evaluation of embeddings |
| 9 | Production: semantic search end-to-end | Embedding index + nearest-neighbor retrieval, saving/loading |

### How to use this notebook
- Cells that train a small model **locally** (gensim Word2Vec on a toy corpus, PCA/t-SNE plots) are left **live and runnable** — no internet needed, they'll execute right here.
- Cells needing a network download (pretrained GloVe, FastText, sentence-transformers) are commented out, exactly like in Modules 1 and 3 — uncomment and run them in an environment with internet.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue from the previous modules.


## 0. Setup

In [ ]:
# %pip install gensim scikit-learn matplotlib sentence-transformers faiss-cpu numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Setup note: uncomment the pip install line above the first time you run this.")


## Part 1 — The Distributional Hypothesis

Every technique in this notebook rests on one 1950s linguistics idea, summarized by J.R. Firth: **"You shall know a word by the company it keeps."** Words that tend to appear in similar contexts tend to have similar meanings. `"cat"` and `"dog"` both show up near words like `"pet"`, `"feed"`, `"vet"` — so a model that learns from context alone ends up placing them near each other in vector space, without ever being told what a cat or a dog *is*.

This is exactly what the co-occurrence matrix in Module 3 Part 4 captured directly (counting) — Word2Vec/GloVe/FastText instead **learn** a compressed vector representation that captures the same contextual signal in far fewer dimensions.


## Part 2 — Word2Vec, Under the Hood

Word2Vec has two architectures, both self-supervised (no manual labels — the training signal comes from the text itself):

- **CBOW (Continuous Bag of Words)**: predict the **center word** from its surrounding **context words**. Faster to train, works well with frequent words.
- **Skip-gram**: predict the surrounding **context words** from the **center word** (the reverse of CBOW). Slower, but generally better for rare words — that's why `sg=1` was the default choice in Module 3.

### Negative sampling (why training is actually fast)
Naively, the network would need a softmax over the *entire* vocabulary at every training step — computationally brutal for a 100k+ word vocabulary. **Negative sampling** turns this into a much cheaper task: for each true (center, context) pair, the model also sees a handful of random, *incorrect* (center, random-word) pairs, and just has to learn to distinguish real pairs from fake ones. This is the trick that made Word2Vec practical at scale.


In [ ]:
# ── Training Word2Vec on a small toy corpus (runs locally, no internet) ─────
# A real project needs a MUCH bigger corpus for meaningful vectors (usually
# millions of words) — this toy corpus exists purely to show the mechanics
# end-to-end. Treat the resulting similarities as illustrative, not accurate.

from gensim.models import Word2Vec

toy_corpus = [
    "the king ruled the kingdom wisely",
    "the queen ruled the kingdom wisely",
    "the man walked into the store",
    "the woman walked into the store",
    "the king and queen visited the castle",
    "the man and woman visited the market",
    "dogs are loyal pets that love their owners",
    "cats are independent pets that love their owners",
    "the dog chased the cat around the garden",
    "the cat chased the mouse around the garden",
]

tokenized_corpus = [sentence.lower().split() for sentence in toy_corpus]

# ── Skip-gram model ──────────────────────────────────────────────────────────
w2v_skipgram = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,     # dimensionality of each word's vector
    window=3,             # context window: 3 words on each side of the center word
    min_count=1,           # keep even words seen only once (real projects use 5+)
    sg=1,                    # 1 = skip-gram, 0 = CBOW
    negative=5,               # negative sampling: 5 "fake" pairs per real pair
    epochs=200,                # small corpus needs many passes to learn anything useful
    seed=42,                    # reproducibility
)

print("Vocabulary:", list(w2v_skipgram.wv.key_to_index.keys()))
print("Vector for 'king':", w2v_skipgram.wv["king"][:5], "... (50 dims total)")


In [ ]:
# ── Exploring what the model learned ─────────────────────────────────────────

print("Words most similar to 'king':")
for word, score in w2v_skipgram.wv.most_similar("king", topn=5):
    print(f"  {word:10s} {score:.3f}")

print()
print("Similarity('king', 'queen') =", round(w2v_skipgram.wv.similarity("king", "queen"), 3))
print("Similarity('king', 'store')  =", round(w2v_skipgram.wv.similarity("king", "store"), 3))
# On a large real corpus, king/queen would score noticeably higher than
# king/store — with this tiny toy corpus the signal is much weaker, but the
# MECHANISM is identical to what you'd see at scale.


In [ ]:
# ── The famous vector-arithmetic analogy trick: king - man + woman ≈ queen ──
# This works because well-trained embeddings capture RELATIONSHIPS as
# consistent directions in vector space, not just individual word meanings.

# result = w2v_skipgram.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
# print(result)
#
# On a small toy corpus like ours this analogy usually won't resolve cleanly —
# it needs a large, diverse corpus to reliably capture such relationships.
# It's shown here as the CLASSIC illustration of what embeddings can encode;
# try it for real once you load pretrained vectors in Part 3.

print("Analogy pattern shown above — most_similar(positive=[...], negative=[...]).")


In [ ]:
# 🔀 CBOW vs Skip-gram — which to actually pick
# | Situation                            | Choose      |
# |----------------------------------------|--------------|
# | Large corpus, want faster training       | CBOW (sg=0)   |
# | Smaller corpus, care about rare words       | Skip-gram (sg=1) |
# | Not sure                                      | Skip-gram — slightly slower but generally more robust |
#
# 📋 COPY-PASTE TEMPLATE for training on YOUR OWN real corpus:
#
# from gensim.models import Word2Vec
# tokenized = [doc.lower().split() for doc in my_documents]   # swap in a real
#                                                              # tokenizer (Module 2)
#                                                              # for production use
# model = Word2Vec(
#     sentences=tokenized,
#     vector_size=100,   # 100-300 is typical; bigger = more expressive, needs more data
#     window=5,
#     min_count=5,        # drop rare words/typos — they get noisy, unreliable vectors anyway
#     sg=1,
#     workers=4,           # parallelize training across CPU cores
#     epochs=10,
# )
# model.save("word2vec_mycorpus.model")            # save for later reuse
# loaded_model = Word2Vec.load("word2vec_mycorpus.model")

print("Guidance + reusable training template shown above.")


## Part 3 — GloVe (Global Vectors)

Word2Vec is **predict-based** — it learns by repeatedly predicting words from local context windows. GloVe is **count-based** — it starts from a giant **global co-occurrence matrix** (exactly like the one you built by hand in Module 3 Part 4, just computed over an entire massive corpus) and factorizes it directly, so the resulting vectors reflect *global* corpus statistics rather than only what's visible through a small sliding window.

In practice: the two methods produce vectors of similar quality, and virtually nobody trains GloVe from scratch — you load **pretrained** GloVe vectors instead.


In [ ]:
# ── Loading pretrained GloVe vectors via gensim's downloader ────────────────

# import gensim.downloader as api
#
# glove_vectors = api.load("glove-wiki-gigaword-100")   # 100-dim, trained on Wikipedia
#                                                          # + Gigaword; downloads once (~130MB),
#                                                          # cached locally after that
#
# print(glove_vectors.most_similar("king"))
# print(glove_vectors.most_similar(positive=["king", "woman"], negative=["man"]))
#
# # On this REAL, large-scale pretrained model, the king-man+woman analogy
# # reliably surfaces "queen" near the top — unlike our tiny toy Word2Vec above.

print("Pretrained GloVe loading pattern shown above — needs internet the first run.")

# 🔀 Other pretrained sizes/sources available via api.load():
# "glove-wiki-gigaword-50/100/200/300", "glove-twitter-25/50/100/200",
# "word2vec-google-news-300" (the original Google News Word2Vec vectors)
# Run `import gensim.downloader as api; list(api.info()['models'].keys())`
# to see the full catalog.


## Part 4 — FastText: Fixing the Out-of-Vocabulary Problem

Word2Vec and GloVe share a hard limitation: if a word wasn't in the training vocabulary, **there is no vector for it at all**. That's a real problem with typos, rare technical terms, or morphologically rich languages (many word forms per root).

**FastText's fix**: represent each word as a bag of **character n-grams** (e.g. `"where"` → `"<wh"`, `"whe"`, `"her"`, `"ere"`, `"re>"`, plus the whole word) and learn a vector for each n-gram. A word's final vector is the sum of its n-gram vectors. This means FastText can build a *reasonable* vector for a word it never saw during training, as long as it shares character pieces with words it did see.


In [ ]:
# ── Training FastText locally (same API shape as Word2Vec) ──────────────────
from gensim.models import FastText

fasttext_model = FastText(
    sentences=tokenized_corpus,   # reusing the toy corpus from Part 2
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    min_n=3,      # minimum character n-gram length
    max_n=6,      # maximum character n-gram length
    epochs=200,
    seed=42,
)

# The real payoff: this word was NEVER in our training corpus, yet FastText
# can still produce a vector for it by combining known character n-grams.
oov_word = "kingdom"   # (present as a substring pattern, but let's try a truly novel one)
novel_word = "kinglike"
print(f"Is '{novel_word}' in the trained vocabulary? {novel_word in fasttext_model.wv.key_to_index}")
print(f"Vector still produced, shape: {fasttext_model.wv[novel_word].shape}")
print("Most similar to the novel word:", fasttext_model.wv.most_similar(novel_word, topn=3))


In [ ]:
# 🔀 FastText vs Word2Vec/GloVe — when to reach for it specifically
# | Situation                                          | Choose FastText? |
# |-------------------------------------------------------|-------------------|
# | Lots of typos/informal text (social media, chat logs)   | Yes                |
# | Morphologically rich language (Finnish, Turkish, Arabic...)| Yes             |
# | Domain with many rare technical terms (medical, legal)     | Yes              |
# | Clean, formal English text, vocabulary is stable              | Word2Vec/GloVe is fine, slightly faster |
#
# 📋 COPY-PASTE TEMPLATE — loading Facebook's official pretrained FastText vectors:
# import gensim.downloader as api
# fasttext_vectors = api.load("fasttext-wiki-news-subwords-300")

print("FastText selection guide shown above.")


## Part 5 — Exploring & Visualizing the Embedding Space

Vectors with 50-300 dimensions can't be plotted directly. **Dimensionality reduction** (PCA or t-SNE) projects them down to 2D purely for visualization, so you can *see* the clusters your embeddings formed.


In [ ]:
# ── PCA projection of our toy Word2Vec vectors to 2D ─────────────────────────
from sklearn.decomposition import PCA

words_to_plot = list(w2v_skipgram.wv.key_to_index.keys())
vectors = np.array([w2v_skipgram.wv[word] for word in words_to_plot])

pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

plt.figure(figsize=(9, 7))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.6)
for word, (x, y) in zip(words_to_plot, vectors_2d):
    plt.annotate(word, (x, y), fontsize=9, xytext=(3, 3), textcoords="offset points")
plt.title("Toy Word2Vec embeddings, PCA-projected to 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

# On such a tiny, low-epoch toy corpus the layout will look fairly noisy —
# on a real corpus you'd typically see clear semantic clusters emerge
# (animals together, royalty-related words together, etc.)


In [ ]:
# 🔀 PCA vs t-SNE for visualization
# | Method | Behavior                                                        |
# |---------|--------------------------------------------------------------------|
# | PCA      | Linear, fast, deterministic, PRESERVES GLOBAL structure/distances    |
# | t-SNE     | Non-linear, slower, stochastic, better at revealing LOCAL clusters, |
# |            | but inter-cluster distances in the plot are NOT meaningful           |
#
# Rule of thumb: use PCA for a quick first look or when you have few points;
# reach for t-SNE (or UMAP) when you have hundreds+ of points and want tight
# local clusters to visually separate.

# from sklearn.manifold import TSNE
# tsne = TSNE(n_components=2, perplexity=5, random_state=42)   # perplexity should be
#                                                                # smaller than your point count
# vectors_2d_tsne = tsne.fit_transform(vectors)
# # ... then plot exactly like the PCA scatter above

print("t-SNE template shown above — swap in for PCA once you have more data points.")


## Part 6 — From Word Vectors to Document Vectors, Properly

A classifier needs one vector *per document*, not one per word. Module 3 showed simple averaging; here's the fuller picture, including a smarter weighting scheme and the dedicated `Doc2Vec` method.


In [ ]:
# ── Method 1: simple averaging (the baseline from Module 3) ──────────────────

def average_word_vectors(document, model, dim):
    words = document.lower().split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

doc_vector = average_word_vectors(toy_corpus[0], w2v_skipgram, dim=50)
print("Averaged document vector shape:", doc_vector.shape)


In [ ]:
# ── Method 2: TF-IDF-WEIGHTED averaging (usually noticeably better) ─────────
# Plain averaging treats every word equally — but "the" and "kingdom" clearly
# shouldn't count the same. Weighting each word's vector by its TF-IDF score
# (Module 3, Part 3) before averaging lets distinctive words dominate the
# resulting document vector, exactly like TF-IDF intended for raw counts.

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf.fit(toy_corpus)
idf_lookup = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def tfidf_weighted_average(document, model, idf_lookup, dim):
    words = document.lower().split()
    weighted_sum = np.zeros(dim)
    total_weight = 0.0
    for word in words:
        if word in model.wv and word in idf_lookup:
            weight = idf_lookup[word]              # rarer words -> higher IDF -> more influence
            weighted_sum += model.wv[word] * weight
            total_weight += weight
    if total_weight == 0:
        return np.zeros(dim)
    return weighted_sum / total_weight

weighted_doc_vector = tfidf_weighted_average(toy_corpus[0], w2v_skipgram, idf_lookup, dim=50)
print("TF-IDF-weighted document vector shape:", weighted_doc_vector.shape)


In [ ]:
# ── Method 3: Doc2Vec — learns document vectors DIRECTLY (not by averaging) ──
# Extends Word2Vec's idea: alongside every word, also learn a vector for each
# whole document (a "paragraph vector"), trained jointly with the word vectors.

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

tagged_docs = [
    TaggedDocument(words=doc.lower().split(), tags=[str(i)])
    for i, doc in enumerate(toy_corpus)
]

doc2vec_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    epochs=200,
    seed=42,
)

# Vector for a document already seen during training:
print("Doc vector for document 0:", doc2vec_model.dv["0"][:5], "...")

# Inferring a vector for a BRAND NEW, unseen document:
new_doc_vector = doc2vec_model.infer_vector("a lion roamed the savanna".lower().split())
print("Inferred vector for a new document:", new_doc_vector[:5], "...")

# 🔀 When to reach for Doc2Vec vs averaging
# Doc2Vec typically captures document-level semantics better than averaging,
# but needs MORE training data to do so reliably. For small/medium corpora,
# TF-IDF-weighted averaging is often the more practical, lower-variance choice.


## Part 7 — Sentence Embeddings (Transformer-Based)

Everything so far builds a document vector out of independent *word* vectors — even weighted averaging never actually reads the sentence as a whole. **Sentence embedding models** (built on BERT-family transformers) instead encode the entire sentence at once, capturing word order and context directly. This is the modern go-to for semantic search, clustering, and similarity in production.

### Pooling strategies (how word-level transformer outputs become ONE sentence vector)
- **[CLS] token pooling**: use the special `[CLS]` token's output vector as the whole sentence's representation (BERT's original design intent, though often outperformed by mean pooling in practice).
- **Mean pooling**: average the transformer's per-token output vectors — usually the strongest general-purpose default, and what `sentence-transformers` uses internally for its most popular models.
- **Max pooling**: take the max value per dimension across tokens — less common, occasionally useful for capturing salient features.


In [ ]:
# ── sentence-transformers: the standard production tool for this ────────────

# %pip install sentence-transformers
#
# from sentence_transformers import SentenceTransformer
#
# model = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, strong general-purpose model
# sentence_vectors = model.encode(toy_corpus)          # shape: (n_docs, 384) — one call,
#                                                        # tokenization + transformer + pooling
#                                                        # all handled internally
#
# from sklearn.metrics.pairwise import cosine_similarity
# sim_matrix = cosine_similarity(sentence_vectors)
# print(pd.DataFrame(sim_matrix, index=toy_corpus, columns=range(len(toy_corpus))).round(2))

print("sentence-transformers usage pattern shown above — needs internet the first run "
      "to download the model (~80MB for all-MiniLM-L6-v2).")

# 🔀 Picking a sentence-transformers model
# | Model                          | Trade-off                                          |
# |------------------------------------|-------------------------------------------------|
# | all-MiniLM-L6-v2                     | Fastest, smallest, great default for most tasks  |
# | all-mpnet-base-v2                     | Slower, noticeably higher quality                  |
# | multi-qa-mpnet-base-dot-v1               | Tuned specifically for search/retrieval (query vs. passage)|
# | paraphrase-multilingual-MiniLM-L12-v2      | Use when your text isn't (only) English            |


## Part 8 — Evaluating Embeddings

You need a way to know if your embeddings are actually good *before* wiring them into a downstream system.

- **Intrinsic evaluation**: test the embedding space directly, independent of any specific application.
  - Word analogy tasks (does `king - man + woman ≈ queen` hold?)
  - Word similarity benchmarks (compare cosine similarity against human-rated similarity scores for word pairs — e.g. the classic WordSim-353 dataset)
- **Extrinsic evaluation**: plug the embeddings into a real downstream task (classification, clustering, search) and measure THAT task's performance. This is what actually matters in production — a high intrinsic score doesn't always translate to a better downstream result.


In [ ]:
# ── A minimal intrinsic evaluation: cosine similarity vs. human intuition ───

def cosine_sim(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

pairs_to_check = [("king", "queen"), ("dog", "cat"), ("king", "store")]
for w1, w2 in pairs_to_check:
    if w1 in w2v_skipgram.wv and w2 in w2v_skipgram.wv:
        sim = cosine_sim(w2v_skipgram.wv[w1], w2v_skipgram.wv[w2])
        print(f"similarity({w1}, {w2}) = {sim:.3f}")

# 📋 COPY-PASTE TEMPLATE for extrinsic evaluation:
# 1. Build document vectors with your embedding method of choice (Part 6/7)
# 2. Feed them into a classifier (sklearn LogisticRegression, as in Module 3 Part 7)
# 3. Compare accuracy/F1 across embedding methods on the SAME train/test split
#    -> whichever method wins THAT comparison is the one to ship, regardless
#       of how it scores on generic word-analogy benchmarks


## Part 9 — Production Template: Semantic Search End-to-End

The single most common production use of embeddings: given a query, find the most semantically similar documents in a collection — this is "semantic search," and it's the retrieval half of modern RAG (retrieval-augmented generation) systems.


In [ ]:
# ── Small-scale: brute-force cosine similarity (fine up to ~100k documents) ─
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, document_vectors, documents, embed_fn, top_k=3):
    """
    📋 COPY-PASTE TEMPLATE
    `embed_fn` is any function that turns a string into a vector — swap in
    average_word_vectors, tfidf_weighted_average, or a sentence-transformers
    .encode() call, and this function works unchanged.
    """
    query_vector = embed_fn(query).reshape(1, -1)
    similarities = cosine_similarity(query_vector, document_vectors)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(documents[i], similarities[i]) for i in top_indices]

doc_vectors = np.array([average_word_vectors(doc, w2v_skipgram, dim=50) for doc in toy_corpus])
results = semantic_search(
    "royalty in the castle",
    doc_vectors,
    toy_corpus,
    embed_fn=lambda text: average_word_vectors(text, w2v_skipgram, dim=50),
)
for doc, score in results:
    print(f"{score:.3f}  {doc}")


In [ ]:
# ── At larger scale: an approximate nearest-neighbor index (FAISS) ──────────
# Brute-force cosine similarity is O(n) per query — fine for thousands of
# documents, too slow for millions. FAISS builds an INDEX that finds
# near-matches in roughly logarithmic time instead.

# import faiss
#
# dimension = doc_vectors.shape[1]
# index = faiss.IndexFlatL2(dimension)     # exact search index; swap for
#                                            # IndexIVFFlat/IndexHNSWFlat at
#                                            # million-scale for approximate,
#                                            # much faster search
# index.add(doc_vectors.astype("float32"))  # FAISS requires float32
#
# query_vector = average_word_vectors("royalty in the castle", w2v_skipgram, dim=50)
# distances, indices = index.search(query_vector.reshape(1, -1).astype("float32"), k=3)
# for idx in indices[0]:
#     print(toy_corpus[idx])

print("FAISS indexing pattern shown above — the standard production tool once your "
      "document collection is too large for brute-force cosine similarity.")

# 🔀 Alternatives at production scale
# - Managed vector databases: Pinecone, Weaviate, Qdrant, Milvus — handle
#   indexing, persistence, filtering, and scaling for you (useful once you
#   don't want to operate FAISS infrastructure yourself)
# - pgvector -> if you're already on PostgreSQL and want vector search
#   without adding a whole new database system


In [ ]:
# ── Saving and loading embedding artifacts for production ───────────────────
import joblib

# joblib.dump(w2v_skipgram, "word2vec_model.joblib")     # or model.save(...) - gensim's
#                                                          # own format, generally preferred
#                                                          # for gensim models specifically
# w2v_skipgram.save("word2vec_model.gensim")
# loaded = Word2Vec.load("word2vec_model.gensim")
#
# np.save("document_vectors.npy", doc_vectors)             # plain numpy save for the
#                                                            # precomputed document vectors
# loaded_doc_vectors = np.load("document_vectors.npy")

print("Save/load patterns shown above — precompute and persist document vectors "
      "once, then only embed NEW incoming queries/documents at request time.")


## Recap & What's Next

You now understand embeddings from the inside out: why the distributional hypothesis makes this all possible, how Word2Vec/GloVe/FastText each solve the problem differently, how to visualize and evaluate an embedding space, how to properly turn word vectors into document vectors (averaging, TF-IDF-weighted averaging, Doc2Vec), and how to build a working semantic search pipeline — from brute-force cosine similarity up to a FAISS index.

### Try this before the next lesson
1. Train a Word2Vec model on your Module 1 corpus (a real one this time, not the toy example) and check whether `most_similar()` returns sensible neighbors for a few words you know well.
2. Build document vectors with TF-IDF-weighted averaging and again with `sentence-transformers`; run the same `semantic_search()` query against both and compare the results.
3. Visualize your trained word vectors with PCA and see if any semantic clusters emerge.

### Next lesson in your NLP mastery path
**Module 5: Classical Machine Learning for NLP** — now that you can turn any text into solid numeric features, we put them to work: Naive Bayes, Logistic Regression, and SVMs for text classification, along with the evaluation metrics (precision/recall/F1, confusion matrices) you need to know whether a model is actually good.
